# Nanograd GPU Acceleration Demo
This notebook demonstrates how to build and train a neural network using Nanograd, supporting both CPU (via NumPy) and GPU (via CuPy/CUDA) dynamically.

The code is hardware-agnostic, meaning the same training loop works on both devices depending on where the inputs are located.

In [1]:
import time
import numpy as np
from nanograd import Tensor
from nanograd.nn import MLP, MSE
from nanograd.optim import Adam

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

print(f"CuPy / CUDA Available: {CUPY_AVAILABLE}")

CuPy / CUDA Available: True


### Synthetic Data Generation

In [2]:
def generate_data(num_samples=2560, input_dim=1024, output_dim=10):
    """Generates synthetic training data."""
    X = np.random.randn(num_samples, input_dim).astype(np.float32)
    W_true = np.random.randn(input_dim, output_dim).astype(np.float32)
    Y = X @ W_true + np.random.randn(num_samples, output_dim).astype(np.float32) * 0.1
    return X, Y

### Hardware-Agnostic Training Loop

In [3]:
def train_model(device='cpu', epochs=10, batch_size=256):
    print(f"\n--- Training on {device.upper()} ---")
    
    # 1. Generate data
    X_np, Y_np = generate_data(num_samples=2560, input_dim=1024, output_dim=10)
    
    # 2. Build model and move to target device (CPU or GPU)
    model = MLP([1024, 1024, 512, 10])
    if device == 'cuda':
        if not CUPY_AVAILABLE:
            print("CUDA/CuPy is not available. Skipping GPU training.")
            return None
        model.cuda()
        
    optimizer = Adam(model.parameters(), learning_rate=0.01)
    loss_fn = MSE()
    if device == 'cuda':
        loss_fn.cuda()

    num_batches = len(X_np) // batch_size
    
    if device == 'cuda':
        cp.cuda.Device().synchronize()

    start_time = time.perf_counter()
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        indices = np.arange(len(X_np))
        np.random.shuffle(indices)
        X_shuffled = X_np[indices]
        Y_shuffled = Y_np[indices]
        
        for b in range(num_batches):
            start_idx = b * batch_size
            end_idx = start_idx + batch_size
            
            # Move mini-batch tensor to the target device dynamically
            x_batch = Tensor(X_shuffled[start_idx:end_idx]).to(device)
            y_batch = Tensor(Y_shuffled[start_idx:end_idx]).to(device)
            
            # Forward & Backward
            predictions = model(x_batch)
            loss = loss_fn(predictions, y_batch)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            loss_val = loss.data
            if hasattr(loss_val, 'get'):
                loss_val = loss_val.get()
            epoch_loss += float(loss_val)
            
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {epoch_loss / num_batches:.4f}")
            
    if device == 'cuda':
        cp.cuda.Device().synchronize()
        
    end_time = time.perf_counter()
    duration = end_time - start_time
    print(f"Training finished in {duration:.4f} seconds.")
    return duration

### Run Benchmarks

In [4]:
print("Running CPU vs GPU Comparison...")
cpu_time = train_model(device='cpu', epochs=10, batch_size=256)
gpu_time = train_model(device='cuda', epochs=10, batch_size=256) if CUPY_AVAILABLE else None

if cpu_time and gpu_time:
    speedup = cpu_time / gpu_time
    print(f"\nSpeedup: {speedup:.2f}x faster on GPU!")

Running CPU vs GPU Comparison...

--- Training on CPU ---
Epoch 01/10 | Loss: 3098515.5300
Epoch 02/10 | Loss: 10143.0801
Epoch 04/10 | Loss: 10143.0801
Epoch 06/10 | Loss: 10143.0801
Epoch 08/10 | Loss: 10143.0801
Epoch 10/10 | Loss: 10143.0801
Training finished in 7.2706 seconds.

--- Training on CUDA ---
Epoch 01/10 | Loss: 1913590.4362
Epoch 02/10 | Loss: 10359.0494
Epoch 04/10 | Loss: 10359.0494
Epoch 06/10 | Loss: 10359.0494
Epoch 08/10 | Loss: 10359.0494
Epoch 10/10 | Loss: 10359.0494
Training finished in 2.1472 seconds.

Speedup: 3.39x faster on GPU!
